# Reto de Laboratorio: Preparando la Matriz Tensorial de la Liga ⚽🤖


Contexto de Negocio:
El equipo de analítica quedó fascinado con el Almacén de Datos (OLAP) que construyeron la semana pasada. Ahora, el departamento de Inteligencia Artificial quiere dar el siguiente paso: entrenar una Red Neuronal para predecir cuántos goles anotará un jugador la próxima temporada.El Científico de Datos les acaba de enviar la tabla resultante del cubo OLAP, pero se detuvo por un error de Scikit-Learn: el algoritmo no sabe leer posiciones como "Forward" ni jerarquías como "Oro".Su misión como Ingenieros de Machine Learning es aplicar Feature Engineering para traducir esta tabla humana a una Matriz $\mathbf{X}$ puramente matemática, sin destruir la lógica vectorial.

In [1]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/players_data-2025_2026.csv"
df_scouting = pd.read_csv(url)

np.random.seed(42)
df_scouting['scout_tier'] = np.random.choice(['Bronce', 'Plata', 'Oro', 'Diamante'], size=len(df_scouting))

if 'player_name' not in df_scouting.columns:
    df_scouting = df_scouting.rename(columns={
        'Player': 'player_name',
        'Squad': 'team_name',
        'Age': 'age',
        'Pos': 'position',
        'Min': 'minutes_played',
        'Gls': 'goals'
    })

columnas_ml = ['player_name', 'team_name', 'age', 'position', 'scout_tier', 'minutes_played', 'goals']
matriz_base = df_scouting[columnas_ml].copy()

print("--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---")
display(matriz_base.head())

--- DATOS BASE PARA EL MODELO (Legibles para humanos) ---


,player_name,team_name,age,position,scout_tier,minutes_played,goals
0,Brenden Aaronson,Leeds United,24.0,"MF,FW",Oro,2449,4
1,Jerome Abbey,Wolves,15.0,MF,Diamante,17,0
2,Zach Abbott,Nottingham Forest,19.0,DF,Bronce,164,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,"MF,FW",Oro,681,2
4,Himad Abdelli,Marseille,25.0,MF,Oro,138,0


### FASE 1: El Traductor Jerárquico (Label Encoding)

Observa la columna `scout_tier`. Contiene valores de texto: 'Bronce', 'Plata', 'Oro', 'Diamante'. Esta variable tiene un **orden natural** claro (Diamante es superior a Bronce).

**Tu tarea:**
Crea un diccionario de mapeo que asigne números enteros respetando la jerarquía natural del talento (iniciando en 1). Luego, aplica este mapeo a la base de datos para crear una nueva columna llamada `scout_encoded`.

In [2]:
jerarquia_scout = {'Bronce': 1, 'Plata': 2, 'Oro': 3, 'Diamante': 4}
matriz_base['scout_encoded'] = matriz_base['scout_tier'].map(jerarquia_scout)

print("Resultado del Label Encoding:")
display(matriz_base[['player_name', 'scout_tier', 'scout_encoded']].head())

Resultado del Label Encoding:


,player_name,scout_tier,scout_encoded
0,Brenden Aaronson,Oro,3
1,Jerome Abbey,Diamante,4
2,Zach Abbott,Bronce,1
3,Jones El-Abdellaoui,Oro,3
4,Himad Abdelli,Oro,3


### FASE 2: El Traductor Nominal (One-Hot Encoding)

Ahora observa la columna `position`. Contiene valores como 'Forward' (Delantero), 'Midfielder' (Mediocampista), 'Defender' (Defensa), etc.
Esta variable es **Nominal**. Si le asignamos números (1 al Delantero y 3 al Defensa), la Red Neuronal creerá falsamente que un Defensa "vale el triple" que un Delantero.

**Tu tarea:**
Utiliza la función de Pandas correspondiente para destruir la columna `position` y expandirla en múltiples columnas booleanas (0 y 1). Guarda el resultado en una nueva variable llamada `matriz_X`.

In [3]:
matriz_X = pd.get_dummies(matriz_base, columns=['position'], dtype=int)

print("Resultado del One-Hot Encoding:")
display(matriz_X.head())

Resultado del One-Hot Encoding:


,player_name,team_name,age,scout_tier,minutes_played,goals,scout_encoded,position_DF,"position_DF,FW","position_DF,MF",position_FW,"position_FW,MF",position_GK,position_MF,"position_MF,DF","position_MF,FW"
0,Brenden Aaronson,Leeds United,24.0,Oro,2449,4,3,0,0,0,0,0,0,0,0,1
1,Jerome Abbey,Wolves,15.0,Diamante,17,0,4,0,0,0,0,0,0,1,0,0
2,Zach Abbott,Nottingham Forest,19.0,Bronce,164,0,1,1,0,0,0,0,0,0,0,0
3,Jones El-Abdellaoui,Celta Vigo,19.0,Oro,681,2,3,0,0,0,0,0,0,0,0,1
4,Himad Abdelli,Marseille,25.0,Oro,138,0,3,0,0,0,0,0,0,1,0,0


### FASE 3: Purgando la Basura Humana

La Inteligencia Artificial está casi lista para devorar los datos. Sin embargo, si le enviamos la matriz ahora mismo, el algoritmo fallará porque todavía hay texto puro estorbando.

**Tu tarea:**
Elimina de `matriz_X` todas las columnas que ya no le sirven al algoritmo:
1. Las columnas de texto original que ya transformaste (como `scout_tier`).
2. Las columnas descriptivas/operativas que no aportan valor matemático universal (`player_name`, `team_name`).

In [4]:
matriz_X = matriz_X.drop(columns=['player_name', 'team_name', 'scout_tier'])

print("--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---")
display(matriz_X.head())
print("\nTipos de datos finales (Todos deben ser numéricos):")
print(matriz_X.dtypes)

--- MATRIZ X DEFINITIVA (Lista para Scikit-Learn / PyTorch) ---


,age,minutes_played,goals,scout_encoded,position_DF,"position_DF,FW","position_DF,MF",position_FW,"position_FW,MF",position_GK,position_MF,"position_MF,DF","position_MF,FW"
0,24.0,2449,4,3,0,0,0,0,0,0,0,0,1
1,15.0,17,0,4,0,0,0,0,0,0,1,0,0
2,19.0,164,0,1,1,0,0,0,0,0,0,0,0
3,19.0,681,2,3,0,0,0,0,0,0,0,0,1
4,25.0,138,0,3,0,0,0,0,0,0,1,0,0



Tipos de datos finales (Todos deben ser numéricos):
age               float64
minutes_played      int64
goals               int64
scout_encoded       int64
position_DF         int64
position_DF,FW      int64
position_DF,MF      int64
position_FW         int64
position_FW,MF      int64
position_GK         int64
position_MF         int64
position_MF,DF      int64
position_MF,FW      int64
dtype: object


### 🧠 FASE 4: Defensa Arquitectónica (Responde aquí mismo)

Imagina que el CTO de la empresa revisa tu código y te hace dos preguntas críticas. Responde con argumentos de ingeniería:

**Pregunta 1:** *"Noté que borraste la columna `team_name` en lugar de aplicarle One-Hot Encoding. En nuestra base de datos mundial de la FIFA tenemos 4,000 equipos diferentes. Si le hubieras aplicado One-Hot Encoding a la columna de equipos, ¿qué le habría pasado a la estructura matemática de tu Matriz X y a la memoria RAM de nuestro servidor?"*
> **Tu Respuesta:** Al aplicar One-Hot Encoding a una columna categórica de alta cardinalidad como `team_name` (con 4,000 equipos distintos), se generarían 4,000 columnas binarias adicionales. Esto desencadenaría la **Maldición de la Dimensionalidad**: la matriz $\mathbf{X}$ se convertiría en una matriz altamente dispersa (*sparse matrix*) compuesta casi en su totalidad por ceros. Como consecuencia, el consumo de memoria RAM se incrementaría de forma desmedida, el costo computacional para entrenar los modelos aumentaría significativamente y existiría un alto riesgo de sobreajuste (*overfitting*).

**Pregunta 2:** *"¿Por qué usamos Label Encoding (1, 2, 3, 4) para la evaluación del scout (`scout_tier`), pero no usamos esa misma técnica matemática para la posición en el campo (`position`)?"*
> **Tu Respuesta:** `scout_tier` es una variable **ordinal**, ya que tiene un orden jerárquico natural ('Diamante' > 'Oro' > 'Plata' > 'Bronce'); Label Encoding asigna números enteros que preservan adecuadamente dicha relación de jerarquía y magnitud. En cambio, `position` es una variable **nominal**, puesto que las distintas posiciones en el terreno de juego no tienen una jerarquía ni orden intrínseco (un delantero no es matemáticamente mayor o menor que un defensa). Aplicar Label Encoding a `position` introduciría un orden numérico arbitrario e inexistente que confundiría y sesgaría al modelo predictivo.